# E13b: Model-choice override record

**Purpose**: record a human-approved override of the E13 rule-based model selection in `model_registry.json`, as a new section `model_choice_override`. Selection is **not** re-run; the locked content (E12) and `oot_evaluation` (E13) are not modified.  
**Inputs**: `config/override.yaml` (human-maintained, read-only), locked `model_registry.json` (all file hashes re-verified), evidence files listed in `evidence_refs`.  
**Outputs**: `model_registry.json` key `model_choice_override` (with timestamp, git commit, config hash), registry backup and run summary under `reports/run_summaries/`, MLflow run.

Rules: every field of `override.yaml` must be filled; `overrides_rule_selection_of` must equal `oot_evaluation.selected_model`; `rules_failed` must equal the rules E13 recorded as failed for the selected model. If the section already exists with different content the module stops: a change needs a new approved entry, kept as history. Later modules read the model with `common.resolve_selected_model(reg)`.

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, shutil, tempfile, hashlib, importlib, copy
from datetime import datetime, date
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import yaml

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

MODULE = "s05b_model_choice_override"
ARTIFACTS = common.artifacts_path(cfg)
REPORTS = os.path.join(ARTIFACTS, "reports")
RUN_SUMS = os.path.join(REPORTS, "run_summaries")
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
OVERRIDE_PATH = os.path.join(PROJECT_ROOT, "config", "override.yaml")
SUMMARY_PATH = os.path.join(RUN_SUMS, f"{MODULE}.json")
os.makedirs(RUN_SUMS, exist_ok=True)

SECTION = common.OVERRIDE_SECTION
FIELDS = common.OVERRIDE_FIELDS
MODEL_KEYS = ("A", "B", "C")
RULE_KEYS = ("R1", "R2", "R3", "R4")

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
RUN_TIME = datetime.now().isoformat(timespec="seconds")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (datetime, date)):
        return o.isoformat()
    return str(o)

def write_json(obj, dst):
    """Local temp file then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(tempfile.mkdtemp(prefix="e13b_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    shutil.copyfile(tmp, dst)

def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=_json_default))

def sha_obj(obj):
    return hashlib.sha256(json.dumps(canonical(obj), sort_keys=True).encode()).hexdigest()

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Registry    : {REGISTRY_PATH} (version {cfg['registry']['version']})")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | run {RUN_TIME}")

In [0]:
assert os.path.exists(OVERRIDE_PATH), f"STOP: {OVERRIDE_PATH} not found"
OVERRIDE_SHA = sha256_file(OVERRIDE_PATH)
with open(OVERRIDE_PATH) as fh:
    OVR_RAW = yaml.safe_load(fh) or {}
assert isinstance(OVR_RAW, dict), "STOP: override.yaml must be a mapping"

def _empty(v):
    return v is None or (isinstance(v, str) and not v.strip()) or (isinstance(v, (list, tuple, dict)) and len(v) == 0)

missing = [f for f in FIELDS if f not in OVR_RAW or _empty(OVR_RAW[f])]
assert not missing, f"STOP: override.yaml has empty or missing fields: {missing}"
unexpected = sorted(set(OVR_RAW) - set(FIELDS))
assert not unexpected, f"STOP: override.yaml has fields outside the agreed list: {unexpected}"

# Normalise types (YAML may parse the date as datetime.date)
OVR = {f: OVR_RAW[f] for f in FIELDS}
OVR["approved_date"] = OVR["approved_date"].isoformat() if isinstance(OVR["approved_date"], (date, datetime)) else str(OVR["approved_date"]).strip()
for f in ("selected_model", "overrides_rule_selection_of", "rationale", "approved_by", "approver_role"):
    OVR[f] = str(OVR[f]).strip()
for f in ("rules_failed", "evidence_refs"):
    assert isinstance(OVR[f], list) and all(not _empty(x) for x in OVR[f]), f"STOP: {f} must be a non-empty list without empty items"
    OVR[f] = [str(x).strip() for x in OVR[f]]

assert OVR["selected_model"] in MODEL_KEYS, f"STOP: selected_model {OVR['selected_model']!r} not in {MODEL_KEYS}"
assert OVR["overrides_rule_selection_of"] in MODEL_KEYS, f"STOP: overrides_rule_selection_of not in {MODEL_KEYS}"
assert OVR["selected_model"] != OVR["overrides_rule_selection_of"], "STOP: selected_model equals the rule selection: nothing to override"
assert set(OVR["rules_failed"]) <= set(RULE_KEYS), f"STOP: rules_failed {OVR['rules_failed']} not within {RULE_KEYS}"
datetime.strptime(OVR["approved_date"], "%Y-%m-%d")    # STOP if not an ISO date

print(f"override.yaml sha256 {OVERRIDE_SHA[:12]} | all {len(FIELDS)} fields filled")
print(f"Override: {OVR['overrides_rule_selection_of']} -> {OVR['selected_model']} | rules failed {OVR['rules_failed']} | "
      f"approved by {OVR['approved_by']} ({OVR['approver_role']}) on {OVR['approved_date']}")

In [0]:
REG0 = read_json(REGISTRY_PATH)
REG0_FILE_SHA = sha256_file(REGISTRY_PATH)
assert REG0.get("locked") is True, "STOP: model_registry.json is not locked"
for k in ("decision_rules_signed_by", "signed_date"):
    assert str(REG0.get("signoff", {}).get(k, "")).strip(), f"STOP: registry sign-off field {k} is empty"
assert REG0.get("version") == cfg["registry"]["version"], f"STOP: registry version {REG0.get('version')} != config {cfg['registry']['version']}"
bad = [k for k, v in REG0["files"].items() if not os.path.exists(v["path"]) or sha256_file(v["path"]) != v["sha256"]]
assert not bad, f"STOP: {len(bad)} locked file(s) changed since the lock: {bad}"
EV = REG0.get("oot_evaluation")
assert EV and EV.get("selected_model"), "STOP: oot_evaluation missing: run E13 (s05_oot) first"

# Hashes BEFORE (acceptance compares them after the write)
LOCKED_SHA_BEFORE = common.registry_locked_sha256(REG0)
OOT_SHA_BEFORE = sha_obj(EV)
assert LOCKED_SHA_BEFORE == EV["locked_content_sha256"], "STOP: locked content differs from the hash E13 recorded"

# The override must refer to what E13 actually recorded
assert OVR["overrides_rule_selection_of"] == EV["selected_model"], (
    f"STOP: override.yaml says it overrides {OVR['overrides_rule_selection_of']}, but E13 selected {EV['selected_model']}")
RULES_SEL = EV["rules"][OVR["selected_model"]]
def _rule_result(r):
    v = RULES_SEL[r]
    if "pass" in v:
        return "PASS" if v["pass"] else "FAIL"
    return str(v.get("status", "UNKNOWN")).upper()
RULE_RESULTS = {r: _rule_result(r) for r in RULE_KEYS}
FAILED_E13 = [r for r in RULE_KEYS if RULE_RESULTS[r] == "FAIL"]
PENDING_E13 = [r for r in RULE_KEYS if RULE_RESULTS[r] not in ("PASS", "FAIL")]
assert sorted(OVR["rules_failed"]) == FAILED_E13, (
    f"STOP: rules_failed {OVR['rules_failed']} != rules E13 recorded as failed for {OVR['selected_model']}: {FAILED_E13}")

# Evidence files must exist ("<path>: <sheet>"; '.xlsx' added when the path as written does not exist)
EVIDENCE = []
for ref in OVR["evidence_refs"]:
    path, sheet = (ref.rsplit(": ", 1) + [None])[:2] if ": " in ref else (ref, None)
    resolved = path if os.path.exists(path) or not os.path.exists(path + ".xlsx") else path + ".xlsx"
    e = {"ref": ref, "path": path, "resolved_path": resolved, "sheet": sheet, "exists": os.path.exists(resolved), "sheet_exists": None, "sha256": None}
    if e["exists"]:
        e["sha256"] = sha256_file(resolved)
        if sheet:
            e["sheet_exists"] = sheet in pd.ExcelFile(resolved).sheet_names
    EVIDENCE.append(e)
bad_ev = [e["ref"] for e in EVIDENCE if not e["exists"] or e["sheet_exists"] is False]
assert not bad_ev, f"STOP: evidence not found: {bad_ev}"

print(f"Registry v{REG0['version']} locked {REG0['locked_at']} | {len(REG0['files'])} file hashes OK")
print(f"Locked sha256 before {LOCKED_SHA_BEFORE[:12]} (= E13 record) | oot_evaluation sha256 before {OOT_SHA_BEFORE[:12]}")
print(f"E13: selected {EV['selected_model']} ({EV['status']}); rule results of {OVR['selected_model']}: {RULE_RESULTS}")
display(pd.DataFrame(EVIDENCE)[["ref", "resolved_path", "sheet", "exists", "sheet_exists"]])

In [0]:
RECORD = dict(OVR)
RECORD.update({
    "recorded_at": RUN_TIME,
    "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH,
    "override_file": {"path": "config/override.yaml", "sha256": OVERRIDE_SHA},
    "registry_version": REG0["version"],
    "locked_content_sha256": LOCKED_SHA_BEFORE,
    "rule_selection": {"selected_model": EV["selected_model"], "status": EV["status"],
                       "selection_reason": EV.get("selection_reason"), "oot_evaluation_run_time": EV.get("run_time")},
    "recorded_rule_results_of_selected": RULE_RESULTS,
    "rules_pending": PENDING_E13,
    "evidence": EVIDENCE,
    "module": MODULE,
})

EXISTING = REG0.get(SECTION)
if EXISTING is None:
    ACTION = "APPEND"
else:
    diff = [f for f in FIELDS if canonical(EXISTING.get(f)) != canonical(OVR[f])]
    if diff:
        raise RuntimeError(
            f"REFUSED: {SECTION} already exists with different content in {diff} "
            f"(recorded {EXISTING.get('recorded_at')}, approved {EXISTING.get('approved_date')} by {EXISTING.get('approved_by')}). "
            "A change needs a new entry with its own approval, kept as history; the existing record is not overwritten.")
    ACTION = "UNCHANGED"           # same approved content: idempotent re-run, keep the original record and timestamp
print(f"Action: {ACTION}")

In [0]:
BACKUP_PATH = None
if ACTION == "APPEND":
    BACKUP_PATH = os.path.join(RUN_SUMS, f"model_registry_before_{SECTION}_{RUN_TIME.replace(':', '').replace('-', '')}.json")
    shutil.copyfile(REGISTRY_PATH, BACKUP_PATH)
    assert sha256_file(BACKUP_PATH) == REG0_FILE_SHA, "backup copy differs from the registry"
    REG1 = copy.deepcopy(REG0)
    REG1[SECTION] = RECORD                     # appended as the last key; nothing else touched
    write_json(REG1, REGISTRY_PATH)
    print(f"{SECTION} appended to {REGISTRY_PATH} | backup {BACKUP_PATH}")
else:
    print(f"{SECTION} already present with the same approved content: registry not rewritten")

REG2 = read_json(REGISTRY_PATH)
SEL = common.resolve_selected_model(REG2)

In [0]:
import mlflow

mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name=MODULE) as run:
    mlflow.set_tags({"module": MODULE, "git_commit": GIT_COMMIT, "registry_version": REG2["version"]})
    mlflow.log_params({"config_sha256": CONFIG_HASH, "override_sha256": OVERRIDE_SHA, "action": ACTION,
                       "selected_model": SEL["model"], "selection_source": SEL["source"],
                       "rule_selected_model": SEL["rule_selected_model"], "rules_failed": ",".join(OVR["rules_failed"]),
                       "approved_by": OVR["approved_by"], "approved_date": OVR["approved_date"],
                       "locked_content_sha256": LOCKED_SHA_BEFORE})
    mlflow.log_artifact(OVERRIDE_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run {MLFLOW_RUN_ID}")

In [0]:
REG3 = read_json(REGISTRY_PATH)
OV3 = REG3.get(SECTION) or {}
CHECKS = {
    "locked content unchanged (sha256 before = after = E13 record)":
        common.registry_locked_sha256(REG3) == LOCKED_SHA_BEFORE == REG3["oot_evaluation"]["locked_content_sha256"],
    "oot_evaluation unchanged (sha256 before = after)": sha_obj(REG3["oot_evaluation"]) == OOT_SHA_BEFORE,
    "no other registry key added, removed or changed":
        canonical({k: v for k, v in REG3.items() if k != SECTION}) == canonical({k: v for k, v in REG0.items() if k != SECTION}),
    "all locked model/binning file hashes still match":
        all(sha256_file(v["path"]) == v["sha256"] for v in REG3["files"].values()),
    f"{SECTION} present": bool(OV3),
    f"{SECTION} complete (all override.yaml fields filled)": all(not _empty(OV3.get(f)) for f in FIELDS),
    f"{SECTION} equals override.yaml": all(canonical(OV3.get(f)) == canonical(OVR[f]) for f in FIELDS),
    f"{SECTION} has timestamp, git commit and config hash": all(not _empty(OV3.get(k)) for k in ("recorded_at", "git_commit", "config_sha256")),
    "override.yaml not modified by this run": sha256_file(OVERRIDE_PATH) == OVERRIDE_SHA,
    "resolve_selected_model returns the override":
        common.resolve_selected_model(REG3, verbose=False)["model"] == OVR["selected_model"] and SEL["source"] == f"{SECTION}.selected_model",
    "oot_evaluation.selected_model still the E13 rule result": REG3["oot_evaluation"]["selected_model"] == EV["selected_model"],
}
RES = pd.DataFrame([{"check": k, "pass": bool(v)} for k, v in CHECKS.items()])
display(RES)
ALL_PASS = bool(RES["pass"].all())

write_json({"module": MODULE, "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
            "registry": {"path": REGISTRY_PATH, "version": REG3["version"], "sha256_before": REG0_FILE_SHA,
                         "sha256_after": sha256_file(REGISTRY_PATH), "backup": BACKUP_PATH},
            "locked_content_sha256": {"before": LOCKED_SHA_BEFORE, "after": common.registry_locked_sha256(REG3)},
            "oot_evaluation_sha256": {"before": OOT_SHA_BEFORE, "after": sha_obj(REG3["oot_evaluation"])},
            "action": ACTION, "override_file_sha256": OVERRIDE_SHA,
            "selected_model": SEL, "rule_results_of_selected": RULE_RESULTS, "rules_pending": PENDING_E13,
            "mlflow_run_id": MLFLOW_RUN_ID, "checks": CHECKS, "all_pass": ALL_PASS,
            "runtime_min": round((time.time() - _t0) / 60, 2)}, SUMMARY_PATH)
failed = RES.loc[~RES["pass"], "check"].tolist()
assert ALL_PASS, f"Acceptance FAILED: {failed}"
print(f"All {len(RES)} acceptance checks PASS | selected model {SEL['model']} ({SEL['source']}) | run summary {SUMMARY_PATH}")